# Historical manager category-profile repeatability

**Question.** Across adjacent completed seasons, is a reviewed manager's *ordering* of category outcomes more similar to their own earlier profile than to profiles created by shuffling manager identities within the same season transition?

This is a narrow descriptive experiment. It does **not** establish manager intent, preference, strategy, skill, causality, a future projection, category scarcity, or a draft recommendation. The purpose is to decide whether there is enough repeatable structure in the completed archive to justify further descriptive investigation.

## The unit and evidence rules

One observation is a **reviewed manager × calendar-consecutive completed-season pair**. A manager is eligible only when the existing domain report has a single reviewed whole-season assignment in both seasons. Shared, dated, ambiguous, unsupported, non-positive-weight, incompatible, or incomplete evidence is excluded rather than treated as zero.

For an eligible pair, the profile is the raw `relative_emphasis` for every compatible category complete in both seasons. Relative emphasis means that category's normalized finish minus the same team's weighted category baseline in that season. The calculation compares the *order* of those values, so category units and season league sizes do not directly set the result.

A transition needs at least six eligible managers. The combined experiment needs at least three such transitions. If either gate fails, the notebook intentionally stops with **INSUFFICIENT COVERAGE**.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import panel as pn

from analysis.workflow import (
    MIN_ELIGIBLE_BLOCKS,
    MIN_MANAGERS_PER_BLOCK,
    PERMUTATION_COUNT,
    PERMUTATION_SEED,
    load_history,
    manager_category_profile_repeatability,
    repeatability_dashboard,
)

# Panel owns the interactive controls and Plotly owns the linked charts.
pn.extension('plotly')

## Run the pre-registered calculation

Spearman correlation is Pearson correlation after ranking each season's category values. Ties receive deterministic average ranks; no SciPy dependency is required. The primary statistic is the median same-manager correlation across all eligible pairs.

The baseline performs 10,000 fixed-seed permutations. In each season-pair block, it independently shuffles only the newer-season manager labels. That preserves the managers, category vectors, transition, and number of correlations while breaking the same-manager pairing. The resulting percentile is a descriptive calibration, **not** a formal independent-sample p-value.

In [ ]:
# The loader invokes the existing read-only archive/domain calculation.
# Missing local archive configuration is a normal, explicit unavailable state.
try:
    context = load_history(ROOT)
    result = manager_category_profile_repeatability(
        context.report,
        permutation_count=PERMUTATION_COUNT,
        permutation_seed=PERMUTATION_SEED,
    )
    view = repeatability_dashboard(result)
except RuntimeError as error:
    result = None
    view = pn.Column(
        pn.pane.Alert(
            f'ARCHIVE UNAVAILABLE: {error}',
            alert_type='warning',
        ),
        pn.pane.Markdown(
            'Configure and import a completed local archive, then rerun this cell. '
            'No data is written by this notebook.'
        ),
    )

# The dashboard links transition and manager selectors to the strip, slope,
# shuffled-baseline ECDF, coverage ledger, and evidence identifiers.
view.servable()

## How to read the linked view

1. Select a season transition. The strip plot shows every eligible manager's same-manager correlation for that transition.
2. Select a manager. The slope view makes stable category ordering and reversals visible; it is the explanation for one correlation, not a separate score.
3. The ECDF shows the 10,000 shuffled pooled medians. The observed line is compared with the shuffled 95th percentile.
4. Inspect the coverage ledger before interpreting the result. The final table preserves observation identifiers, mapper versions, and assignment revisions for the selected profile.

The controls are intentionally limited. They help inspect the evidence behind a fixed experiment rather than invite additional dashboard metrics.

## Pre-committed decision rule and stopping point

Report descriptive aggregate repeatability only when **both** conditions hold:

- the observed pooled median is greater than the shuffled 95th percentile; and
- at least half of eligible transitions have a positive median same-manager correlation.

Otherwise record: **No robust aggregate category-profile repeatability was found.** Stop expanding manager-preference visualizations; do not tune the threshold, add a more complex model, or recast a weak result as a strategy conclusion.

## Limitations

A repeatable outcome profile can reflect roster continuity, injuries, transactions, league context, or chance. The archive has a small number of categories, so ties are common. Season transitions and managers are reused, so the shuffled comparison is not a formal p-value. Complete-vector eligibility can favor better-documented managers. A weak aggregate result can coexist with individual profiles worth describing, but it does not justify inference about intent.

Before committing this notebook, clear all cell outputs. Committed source contains only this explanation and executable code; real archive data and rendered findings stay local.